In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import utulek

utulek.platform.import_globals_notebook(globals())

base_module = utulek


I0000 00:00:1785182963.490598  789448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785182963.491075  789448 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785182965.000006  789448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785182965.000305  789448 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


IS_NOTEBOOK_KERNEL_CODE = True
NOTEBOOK_NAME = 2026_07_27-t2t.ipynb
ASSET_PATH = /home/gilgamesh/main.syncthing/utulek/experiment/2026_07_27-t2t.ipynb.asset/
Loaded `/home/gilgamesh/main.syncthing/utulek/experiment/.env`.
torch: []
tf: 
jax: [CpuDevice(id=0)]


E0000 00:00:1785182967.582585  789448 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint,
	dtype="auto",
	device_map="auto")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [4]:
tokenizer.response_schema = {
	"x-regex":
	r"^(?:(?:<think>)?\s*(?P<thinking>.+?)\s*</think>)?\s*(?:<tool_call>(?P<tool_calls>.*?)\s*</tool_call>)?\s*(?P<content>.+?)?\s*(?:<\|im_end\|>|$)",
	"type": "object",
	"properties": {
	"role": {
	"const": "assistant"
	},
	"content": {
	"type": "string"
	},
	"thinking": {
	"type": "string"
	},
	"tool_calls": {
	"x-regex-iterator": r"^(.*)$",
	"type": "array",
	"items": {
	"type": "object",
	"properties": {
	"type": {
	"const": "function"
	},
	"function": {
	"x-parser": "json",
	"x-parser-args": {
	"allow_non_json": True
	},
	"type": "object",
	"properties": {
	"name": {
	"type": "string"
	},
	"arguments": {
	"type": "object",
	"additionalProperties": {}
	},
	},
	},
	},
	},
	},
	},
}

In [5]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(user_prompt: str,
	history=[{
	"role":
	"system",
	"content":
	"You are a general-purpose technical assistant."
	}],
	max_new_tokens: int = 8192,
	enable_thinking: bool = False):
	messages = history
	messages.append({"role": "user", "content": user_prompt})
	input_ids = tokenizer.apply_chat_template(messages,
		add_generation_prompt=True,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"].to(model.device)
	streamer = TextIteratorStreamer(tokenizer,
		skip_prompt=True,
		skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0.0,
		# "presence_penalty": 0.0,
		"repetition_penalty": 1.0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(target=model.generate,
		kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append({
		"role": "assistant",
		"content": "".join(tokens)
	})


history = [{
	"role":
	"system",
	"content":
	"You are a general-purpose technical assistant."
}]

In [6]:
model.to("cpu")

for token in prompt(
		"why does salting meat make it last longer",
		history=history,
		enable_thinking=False):
	print(token, end="")

with open("../snowfall/t2t.md", "w") as f:
	f.write(history[-1]["content"])

Salting meat is one of the oldest and most effective preservation methods because it creates an environment that is hostile to bacteria, fungi, and other microorganisms responsible for spoilage. The process works through a combination of **osmosis**, **dehydration**, and the creation of a **hypertonic environment**.

Here is the step-by-step breakdown of how it works:

### 1. Osmosis and Water Removal
When salt (sodium chloride) is applied to meat, the concentration of salt in the surface layer becomes extremely high compared to the concentration of water inside the muscle cells.
*   **Osmotic Pressure**: Water naturally moves from areas of low solute concentration (inside the meat) to areas of high solute concentration (the salt on the surface) to equalize the balance. This process is called osmosis.
*   **Dehydration**: As water rushes out of the meat cells and evaporates, the meat becomes dehydrated. This reduces the overall moisture content available for bacteria to survive and mul

FileNotFoundError: [Errno 2] No such file or directory: '../snowfall/t2t.md'

In [21]:
model.to("cpu")
torch.cuda.empty_cache()